In [1]:
import pandas as pd

In [2]:
df_jan = pd.read_parquet("https://nyc-tlc.s3.amazonaws.com/trip+data/fhv_tripdata_2021-01.parquet")
df_feb = pd.read_parquet("https://nyc-tlc.s3.amazonaws.com/trip+data/fhv_tripdata_2021-02.parquet")

In [3]:
print(f"The data for January have {df_jan.shape[0]} records")

The data for January have 1154112 records


In [4]:
df_jan.head()

,dispatching_base_num,pickup_datetime,dropOff_datetime,PUlocationID,DOlocationID,SR_Flag,Affiliated_base_number
0,B00009,2021-01-01 00:27:00,2021-01-01 00:44:00,NaN,NaN,None,B00009
1,B00009,2021-01-01 00:50:00,2021-01-01 01:07:00,NaN,NaN,None,B00009
2,B00013,2021-01-01 00:01:00,2021-01-01 01:51:00,NaN,NaN,None,B00013
3,B00037,2021-01-01 00:13:09,2021-01-01 00:21:26,NaN,72.0,None,B00037
4,B00037,2021-01-01 00:38:31,2021-01-01 00:53:44,NaN,61.0,None,B00037


In [5]:
df_jan["duration_minutes"] = (df_jan["dropOff_datetime"] - df_jan["pickup_datetime"]).astype('timedelta64[m]')

In [6]:
avg_duration = df_jan["duration_minutes"].mean()
print(f"The average trip duration is {avg_duration} minutes")

The average trip duration is 18.733049305440026 minutes


In [7]:
df_jan_filtered  = df_jan[df_jan["duration_minutes"].between(1, 60, inclusive="both")]

In [8]:
print(f"The average trip duration is {df_jan_filtered['duration_minutes'].mean()} minutes")

The average trip duration is 15.8499162370496 minutes


In [9]:
# import seaborn as sns
# sns.histplot(df_jan["duration_minutes"])

In [10]:
df_jan_filtered["PUlocationID"].fillna(value=-1, inplace=True)
df_jan_filtered["DOlocationID"].fillna(value=-1, inplace=True)

C:\Users\papageorge\AppData\Local\Temp\ipykernel_11848\3867599641.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jan_filtered["PUlocationID"].fillna(value=-1, inplace=True)
C:\Users\papageorge\AppData\Local\Temp\ipykernel_11848\3867599641.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jan_filtered["DOlocationID"].fillna(value=-1, inplace=True)


In [11]:
missing_pickup_location_perc = (df_jan_filtered["PUlocationID"] == -1).sum() / df_jan_filtered.shape[0]
print(f"The fraction of missing for the pick up location ID is {missing_pickup_location_perc*100}%")

The fraction of missing for the pick up location ID is 83.51935819846193%


In [12]:
from sklearn.feature_extraction import DictVectorizer

df_jan_filtered["PUlocationID"] = df_jan_filtered["PUlocationID"].astype(str)
df_jan_filtered["DOlocationID"] = df_jan_filtered["DOlocationID"].astype(str)

C:\Users\papageorge\AppData\Local\Temp\ipykernel_11848\790358367.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jan_filtered["PUlocationID"] = df_jan_filtered["PUlocationID"].astype(str)
C:\Users\papageorge\AppData\Local\Temp\ipykernel_11848\790358367.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_jan_filtered["DOlocationID"] = df_jan_filtered["DOlocationID"].astype(str)


In [13]:
features = ["PUlocationID", "DOlocationID"]
list_of_dicts = df_jan_filtered[features].to_dict(orient="records")

In [14]:
v = DictVectorizer()
X_train = v.fit_transform(list_of_dicts)

In [15]:
X_train

<1110873x525 sparse matrix of type '<class 'numpy.float64'>'
	with 2221746 stored elements in Compressed Sparse Row format>

In [16]:
from sklearn.linear_model import LinearRegression

In [17]:
lr = LinearRegression().fit(X=X_train, y=df_jan_filtered["duration_minutes"])

In [18]:
y_pred_train = lr.predict(X_train)

In [19]:
from sklearn.metrics import mean_squared_error
mean_squared_error(df_jan_filtered["duration_minutes"], y_pred_train, squared=False)

10.59522216378258

In [20]:
df_feb.head()

,dispatching_base_num,pickup_datetime,dropOff_datetime,PUlocationID,DOlocationID,SR_Flag,Affiliated_base_number
0,B00013,2021-02-01 00:01:00,2021-02-01 01:33:00,NaN,NaN,None,B00014
1,B00021,2021-02-01 00:55:40,2021-02-01 01:06:20,173.0,82.0,None,B00021
2,B00021,2021-02-01 00:14:03,2021-02-01 00:28:37,173.0,56.0,None,B00021
3,B00021,2021-02-01 00:27:48,2021-02-01 00:35:45,82.0,129.0,None,B00021
4,B00037,2021-02-01 00:12:50,2021-02-01 00:26:38,NaN,225.0,None,B00037


In [21]:
df_feb["duration_minutes"] = (df_feb["dropOff_datetime"] - df_feb["pickup_datetime"]).astype('timedelta64[m]')

In [22]:
df_feb["PUlocationID"] = df_feb["PUlocationID"].astype(str)
df_feb["DOlocationID"] = df_feb["DOlocationID"].astype(str)

In [23]:
list_of_dicts_validation = df_feb[features].to_dict(orient="records")

In [24]:
X_val = v.transform(list_of_dicts_validation)

In [25]:
y_pred_val = lr.predict(X_val)

In [26]:
mean_squared_error(df_feb["duration_minutes"], y_pred_val, squared=False)

161.00425788749465